# Sorcery TCG Card Scanner - Model Training

This notebook trains a MobileNetV2-based classifier to recognize Sorcery TCG cards.

**Instructions:**
1. Run all cells in order
2. Upload your training data when prompted (from `data/scanner-training/images.zip`)
3. Download the trained model at the end

**Requirements:**
- Enable GPU: Runtime → Change runtime type → T4 GPU

In [ ]:
# Install dependencies
!pip install -q tensorflowjs

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import json
import os

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## Upload Training Data

Upload your `images.zip` file created by:
```bash
cd data/scanner-training && zip -r images.zip images/
```

In [ ]:
from google.colab import files
import zipfile

# Upload the images.zip file
uploaded = files.upload()

# Extract
with zipfile.ZipFile('images.zip', 'r') as zip_ref:
    zip_ref.extractall('data')

print("Extracted to data/images/")

In [ ]:
# Configuration
DATA_DIR = 'data/images'
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 20

# Count classes
num_classes = len(os.listdir(DATA_DIR))
print(f"Found {num_classes} classes")

In [ ]:
# Data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    brightness_range=[0.8, 1.2],
    horizontal_flip=False,
    fill_mode='nearest',
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

validation_generator = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")

In [ ]:
# Build model
base_model = MobileNetV2(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights='imagenet',
    pooling='avg'
)

base_model.trainable = False

model = keras.Sequential([
    base_model,
    layers.Dropout(0.3),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy', 'top_k_categorical_accuracy']
)

model.summary()

In [ ]:
# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6
    )
]

# Phase 1: Train head only
print("Phase 1: Training classification head...")
history1 = model.fit(
    train_generator,
    epochs=EPOCHS // 2,
    validation_data=validation_generator,
    callbacks=callbacks
)

In [ ]:
# Phase 2: Fine-tune top layers
print("Phase 2: Fine-tuning base model...")
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy', 'top_k_categorical_accuracy']
)

history2 = model.fit(
    train_generator,
    epochs=EPOCHS // 2,
    validation_data=validation_generator,
    callbacks=callbacks
)

In [ ]:
# Evaluate
print("\nEvaluating model...")
results = model.evaluate(validation_generator)
print(f"Validation Loss: {results[0]:.4f}")
print(f"Validation Accuracy: {results[1]:.4f}")
print(f"Top-5 Accuracy: {results[2]:.4f}")

In [ ]:
# Save class map
class_indices = train_generator.class_indices
index_to_class = {v: k for k, v in class_indices.items()}

class_map = {
    'classIndices': class_indices,
    'indexToClass': index_to_class,
    'numClasses': num_classes
}

with open('class_map.json', 'w') as f:
    json.dump(class_map, f, indent=2)

print("Saved class_map.json")

In [ ]:
# Convert to TensorFlow.js
import tensorflowjs as tfjs

os.makedirs('tfjs_model', exist_ok=True)
tfjs.converters.save_keras_model(model, 'tfjs_model')

# Copy class map
import shutil
shutil.copy('class_map.json', 'tfjs_model/class_map.json')

print("Model saved to tfjs_model/")

In [ ]:
# Zip for download
!zip -r card_scanner_model.zip tfjs_model/

from google.colab import files
files.download('card_scanner_model.zip')

print("\nDownload complete! Extract to public/models/card-scanner/tfjs/")